In [ ]:
import sqlite3
import struct
from datetime import datetime

DB_PATH = "/data/cat.db"

def safe_float(v):
    if v is None:
        return 0.0
    if isinstance(v, bytes):
        try:
            return struct.unpack('<d', v)[0] if len(v) == 8 else 0.0
        except:
            return 0.0
    try:
        return float(v)
    except:
        return 0.0

def ts_to_human(ts):
    ts = safe_float(ts)
    unit = 1000.0 if ts > 1e11 else 1.0
    try:
        return datetime.fromtimestamp(ts / unit).strftime('%Y/%m/%d %H:%M:%S')
    except:
        return f"変換失敗({ts})"

conn = sqlite3.connect(DB_PATH, timeout=30)

print("=== labelsテーブル 直近10件 ===")
rows = conn.execute(
    "SELECT start_ts, end_ts, label, cat_w, waste_w FROM labels ORDER BY start_ts DESC LIMIT 10"
).fetchall()

if not rows:
    print("  ラベルが1件もありません")
else:
    for r in rows:
        print(f"  {ts_to_human(r[0])}  label={r[2]}  cat_w={r[3]}g  waste_w={r[4]}g")

print()
print(f"=== labels総件数: {conn.execute('SELECT COUNT(*) FROM labels').fetchone()[0]}件 ===")

conn.close()